# Spatial GRN Integration of the Dentate Gyrus

## Step2:Gene Regulatory Network uisng SCENIC+:
## Tutorial 2c: SCENIC+

**Author:** Xuan Cao  
**Affiliation:** Icahn School of Medicine at Mount Sinai  
**Email:** xuan.cao@mssm.edu  

**Contributors:** Donghoon Lee, Panos Roussos  

**Date:** March 20, 2026  

---

### Overview
This notebook demonstrates a pipeline for spatial imputation and integration of gene regulatory networks (GRNs) in the dentate gyrus. Using Tangram, we map single-cell transcriptomic profiles onto spatial transcriptomics data to enable downstream GRN inference in a spatially resolved context.

### scenicplus version 
0.1.dev454+g9320635

### Install conda environment
Adapted from the SCENIC+ GitHub repository<br>
conda create --name scenicplus python=3.11 -y<br>
conda activate scenicplus<br>
git clone https://github.com/aertslab/scenicplus<br>
cd scenicplus<br>
pip install .<br>


### Contents
1. Data loading 
2. Generate pseudobulk ATAC-seq profiles  
3. Peak Calling  
4. Creating cisTopic object
5. Running topic model
6. Visualization of topics
7. Topic binarization & QC
8. Differentially Accessible Regions (DARs)
9. Export cistopic object
  

# Imports

In [1]:
import numpy as np
import scanpy as sc
import pandas as pd
import pyranges as pr

sc.set_figure_params(scanpy=True, dpi=100, dpi_save=150, fontsize=12)
sc.set_figure_params(dpi=100, color_map = 'viridis_r')
sc.settings.verbosity = 1
sc.logging.print_header()
from matplotlib.pyplot import rc_context

import gc

import sys
import os
import pickle

import warnings
warnings.simplefilter(action='ignore')
import scenicplus
scenicplus.__version__

/sc/arion/projects/YuanLab/gcproj/xuan/anaconda3/envs/scenicplus/lib/python3.8/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/sc/arion/projects/YuanLab/gcproj/xuan/anaconda3/envs/scenicplus/lib/python3.8/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/sc/arion/pr

scanpy==1.8.2 anndata==0.8.0 umap==0.5.3 numpy==1.23.5 scipy==1.9.3 pandas==1.5.1 scikit-learn==1.3.2 statsmodels==0.13.5 python-igraph==0.10.2 pynndescent==0.5.11


/sc/arion/projects/YuanLab/gcproj/xuan/anaconda3/envs/scenicplus/lib/python3.8/site-packages/umap/umap_.py:660: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()


'0.1.dev454+g9320635'

# Set dirs

In [2]:
master_dir ='SpatialGRN/Output'
cistopic_dir = f'{master_dir}/02a.pycisTopic'
cistarget_dir = f'{master_dir}/02b.pycisTarget'
scplus_dir = f'{master_dir}/02c.scenic+'

tmp_dir = "tmp/"
log_dir=f'{cistopic_dir}/00.log'
run_dir = f'{scplus_dir}/00.run'
 
if not os.path.exists(scplus_dir):
    os.makedirs(scplus_dir)
    
if not os.path.exists(tmp_dir):
    os.makedirs(tmp_dir)

if not os.path.exists(tmp_dir):
    os.makedirs(log_dir)

# 1. build scenicplus object

In [3]:
# Input data path
rna_h5ad = 'data/human_DG_snrna.h5ad'
cistopic_path = f'{cistopic_dir}/cistopic_obj.pkl'
menr_path = f'{cistarget_dir}/menr.pkl'

# Load data
rna = sc.read_h5ad(rna_h5ad)
cistopic_obj = pickle.load(open(cistopic_path, 'rb'))
menr = pickle.load(open(menr_path, 'rb'))
# change the cellbarcode to the same form
rna.obs_names = rna.obs_names + '___human_DG'

In [14]:
#######
from pycisTopic.cistopic_class import *
from scenicplus.scenicplus_class import SCENICPLUS, create_SCENICPLUS_object

# create scenicplus object
scplus_obj = create_SCENICPLUS_object(
    GEX_anndata = rna,
    cisTopic_obj = cistopic_obj,
    menr = menr,
)
scplus_obj.X_EXP = scplus_obj.X_EXP.toarray()
print(scplus_obj)
print("1.create_SCENICPLUS_object done!")

2026-06-13 10:40:19,959 cisTopic     INFO     Imputing drop-outs
2026-06-13 10:40:31,814 cisTopic     INFO     Scaling
2026-06-13 10:40:43,028 cisTopic     INFO     Keep non zero rows
2026-06-13 10:40:56,265 cisTopic     INFO     Imputed accessibility sparsity: 0.5326049393648504
2026-06-13 10:40:56,268 cisTopic     INFO     Create CistopicImputedFeatures object
2026-06-13 10:40:56,269 cisTopic     INFO     Done!
SCENIC+ object with n_cells x n_genes = 9856 x 36601 and n_cells x n_regions = 9856 x 594681
	metadata_regions:'Chromosome', 'Start', 'End', 'Width', 'cisTopic_nr_frag', 'cisTopic_log_nr_frag', 'cisTopic_nr_acc', 'cisTopic_log_nr_acc'
	metadata_cell:'GEX_class', 'GEX_subclass', 'GEX_subtype', 'ACC_cisTopic_nr_frag', 'ACC_cisTopic_log_nr_frag', 'ACC_cisTopic_nr_acc', 'ACC_cisTopic_log_nr_acc', 'ACC_sample_id', 'ACC_Unique_nr_frag', 'ACC_barcode', 'ACC_class', 'ACC_subclass', 'ACC_subtype'
	menr:'CTX_topics_otsu_All', 'CTX_topics_otsu_No_promoters', 'DEM_topics_otsu_All', 'DEM_t

# 2. Merge cistromes (all)

In [15]:
# 2. Merge cistromes (all)

from scenicplus.cistromes import *
merge_cistromes(scplus_obj)

tfs = list(scplus_obj.uns['Cistromes']['Unfiltered'].keys())
print('tf numbers from motif dir',len(tfs))
print(scplus_obj.uns['Cistromes']['Unfiltered'][tfs[0]])

cistromes = scplus_obj.uns['Cistromes']  
with open(scplus_dir + '/02.Cistromes.pkl', 'wb') as f:
  pickle.dump(cistromes, f)
  
print("2. Merge cistromes done!")


tf numbers from motif dir 2349
+--------------+-----------+-----------+
| Chromosome   | Start     | End       |
| (category)   | (int32)   | (int32)   |
|--------------+-----------+-----------|
| chr1         | 8987871   | 8988371   |
| chr1         | 201042804 | 201043304 |
| chr1         | 230073510 | 230074010 |
| chr1         | 87393468  | 87393968  |
| ...          | ...       | ...       |
| chrX         | 3697647   | 3698147   |
| chrX         | 103782596 | 103783096 |
| chrX         | 101415560 | 101416060 |
| chrX         | 123764982 | 123765482 |
+--------------+-----------+-----------+
Unstranded PyRanges object has 3,248 rows and 3 columns from 23 chromosomes.
For printing, the PyRanges was sorted on Chromosome.
2. Merge cistromes done!


# 3. Infer enhancer to gene relationships

## 3A. get gene space

In [16]:
annot = pickle.load(open('resources/hg38_annot.get_search_space.pkl', 'rb'))
chromszies_file = "resources/hg38.chrom.sizes"
upstream = [1000, 150000]
downstream = [1000, 150000]

# read chromsize
import pandas as pd
import pyranges as pr

if not any(['chr' in c for c in scplus_obj.region_names]):
    annot.Chromosome = annot.Chromosome.str.replace('chr', '')

chromsizes = pd.read_csv(chromszies_file, sep='\t', header=None)
chromsizes.columns = ['Chromosome', 'End']
chromsizes['Start'] = [0]*chromsizes.shape[0]
chromsizes = chromsizes.loc[:, ['Chromosome', 'Start', 'End']]
if not any(['chr' in c for c in scplus_obj.region_names]):
    annot.Chromosome = annot.Chromosome.str.replace('chr', '')
chromsizes = pr.PyRanges(chromsizes)

#------------------------------------
from scenicplus.enhancer_to_gene import get_search_space, calculate_regions_to_genes_relationships, GBM_KWARGS
get_search_space(scplus_obj,
                 pr_annot = annot,
                 pr_chromsizes = chromsizes,
                 upstream = upstream,
                 downstream = downstream)

search_space = scplus_obj.uns['search_space']
with open(scplus_dir + '/03.search_space.pkl', 'wb') as f:
  pickle.dump(search_space, f)

print(search_space)
print(scplus_obj.uns.keys())

print("3A.Get search space done!")

2026-06-13 10:44:20,546 R2G          INFO     Extending promoter annotation to 10 bp upstream and 10 downstream
2026-06-13 10:44:22,219 R2G          INFO     Extending search space to:
            						150000 bp downstream of the end of the gene.
            						150000 bp upstream of the start of the gene.
2026-06-13 10:44:35,262 R2G          INFO     Intersecting with regions.


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


2026-06-13 10:44:36,237 R2G          INFO     Calculating distances from region to gene
2026-06-13 10:47:39,049 R2G          INFO     Imploding multiple entries per region and gene
2026-06-13 10:52:34,759 R2G          INFO     Done!
                           Name     Gene   Distance
0        chr1:12565881-12566381  AADACL3   [149979]
1        chr1:12567468-12567968  AADACL3   [148392]
2        chr1:12573022-12573522  AADACL3   [142838]
3        chr1:12573707-12574207  AADACL3   [142153]
4        chr1:12581110-12581610  AADACL3   [134750]
...                         ...      ...        ...
2111193  chrY:13479199-13479699      UTY        [0]
2111194  chrY:13479821-13480321      UTY        [0]
2111195  chrY:13493456-13493956      UTY    [13033]
2111196  chrY:13543836-13544336      UTY    [63413]
2111197  chrY:13852133-13852633      VCY  [-133389]

[2111198 rows x 3 columns]
dict_keys(['Cistromes', 'search_space'])
3A.Get search space done!


## 3B.Region-to-gene calculate

In [17]:
# calculate region_to_gene relationship
from scenicplus.enhancer_to_gene import get_search_space, calculate_regions_to_genes_relationships, GBM_KWARGS
ncpu = 10

calculate_regions_to_genes_relationships(
    scplus_obj,
    ray_n_cpu = ncpu,
    _temp_dir = tmp_dir,
    importance_scoring_method = 'GBM',
    importance_scoring_kwargs = GBM_KWARGS
)

2026-06-13 11:26:28,710 R2G          INFO     Calculating region to gene importances, using GBM method


2026-06-13 11:26:42,938	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
Running using 10 cores: 100%|██████████| 19206/19206 [21:14<00:00, 15.07it/s]  


2026-06-13 12:18:29,475 R2G          INFO     Took 3120.7636647224426 seconds
2026-06-13 12:18:29,477 R2G          INFO     Calculating region to gene correlation, using SR method


2026-06-13 12:18:37,518	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
(_score_regions_to_single_gene_ray pid=2881520) /sc/arion/projects/YuanLab/gcproj/xuan/anaconda3/envs/scenicplus/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4881: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
(_score_regions_to_single_gene_ray pid=2881520)   warnings.warn(stats.ConstantInputWarning(warn_msg))
(_score_regions_to_single_gene_ray pid=2881520) /sc/arion/projects/YuanLab/gcproj/xuan/anaconda3/envs/scenicplus/lib/python3.8/site-packages/scipy/stats/_stats_py.py:4881: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
(_score_regions_to_single_gene_ray pid=2881520)   warnings.warn(stats.ConstantInputWarning(warn_msg))
(_score_regions_to_single_gene_ray pid=2881521) /sc/arion/projects/YuanLab/gcproj/xuan/anaconda3/envs/scenicplus/lib/python3.8/site-packages/s

2026-06-13 12:47:33,977 R2G          INFO     Took 1744.4988021850586 seconds
2026-06-13 12:47:45,382 R2G          INFO     Done!


# 4. Infer TF to gene relationships

In [29]:
from scenicplus.TF_to_gene import *
tf_file='resources/utoronto_human_tfs_v_1.01.txt'
ncpu = 20

calculate_TFs_to_genes_relationships(
    scplus_obj,
    tf_file = tf_file,
    ray_n_cpu = ncpu,
    method = 'GBM',
    _temp_dir = tmp_dir,
    key= 'TF2G_adj'
)

2026-06-13 18:03:43,963	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8266 


2026-06-13 18:03:45,281 TF2G         INFO     Calculating TF to gene correlation, using GBM method


(raylet) Spilled 7188 MiB, 15 objects, write throughput 2494 MiB/s. Set RAY_verbose_spill_logs=0 to disable this message.
(raylet) Spilled 12938 MiB, 27 objects, write throughput 3277 MiB/s.
(raylet) Spilled 34442 MiB, 70 objects, write throughput 2676 MiB/s. [repeated 3x across cluster]
(raylet) Spilled 133639 MiB, 277 objects, write throughput 3665 MiB/s. [repeated 2x across cluster]
(raylet) Spilled 267922 MiB, 546 objects, write throughput 4207 MiB/s.
(raylet) Spilled 525536 MiB, 1067 objects, write throughput 5671 MiB/s.
(raylet) Spilled 1049727 MiB, 2112 objects, write throughput 5775 MiB/s.
(raylet) Spilled 2100434 MiB, 4240 objects, write throughput 5768 MiB/s.
(raylet) Spilled 4194475 MiB, 8478 objects, write throughput 5715 MiB/s.
(raylet) Spilled 8390755 MiB, 17005 objects, write throughput 5727 MiB/s.
(raylet) Spilled 16778637 MiB, 34006 objects, write throughput 5753 MiB/s.
Running using 20 cores: 100%|██████████| 36601/36601 [04:47<00:00, 127.48it/s] 


2026-06-13 19:42:59,681 TF2G         INFO     Took 5954.397935152054 seconds
2026-06-13 19:42:59,683 TF2G         INFO     Adding correlation coefficients to adjacencies.
2026-06-13 19:45:12,764 TF2G         INFO     Warning: adding TFs as their own target to adjecencies matrix. Importance values will be max + 1e-05
2026-06-13 19:45:19,006 TF2G         INFO     Adding importance x rho scores to adjacencies.
2026-06-13 19:45:19,033 TF2G         INFO     Took 139.350195646286 seconds


# 5. Build GRN

In [ ]:
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ': Start build GRN.',sep="")
# Load functions
from scenicplus.grn_builder.gsea_approach import build_grn
build_grn(scplus_obj,
          min_target_genes = 10,
          adj_pval_thr = 1,
          min_regions_per_gene = 0,
          quantiles = (0.85, 0.90, 0.95),
          top_n_regionTogenes_per_gene = (5, 10, 15),
          top_n_regionTogenes_per_region = (),
          binarize_using_basc = True,
          rho_dichotomize_tf2g = True,
          rho_dichotomize_r2g = True,
          rho_dichotomize_eregulon = True,
          rho_threshold = 0.001,
          keep_extended_motif_annot = True,
          merge_eRegulons = True,
          order_regions_to_genes_by = 'importance',
          order_TFs_to_genes_by = 'importance',
          key_added = 'eRegulons_importance',
          cistromes_key = 'Unfiltered',
          disable_tqdm = False, #If running in notebook, set to True
          ray_n_cpu = 40,
          _temp_dir = tmp_dir)

print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':End build GRN.',sep="")


2026-06-13 20:14:10: Start build GRN.
2026-06-13 20:14:13,753 GSEA         INFO     Thresholding region to gene relationships


2026-06-13 20:14:29,072	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8266 
 21%|██▏       | 3/14 [07:37<25:42, 140.20s/it]                                             
2;1mProcessing: negative r2g, Top 5 region-to-gene links per gene:   0%|          | 0/2349 [00:00<?, ?it/s]
2;1mProcessing: negative r2g, Top 5 region-to-gene links per gene:   0%|          | 3/2349 [00:00<01:37, 24.03it/s]
2;1mProcessing: negative r2g, Top 5 region-to-gene links per gene:   0%|          | 6/2349 [00:00<01:28, 26.35it/s]
2;1mProcessing: negative r2g, Top 5 region-to-gene links per gene:   0%|          | 10/2349 [00:00<01:18, 29.81it/s]
2;1mProcessing: negative r2g, Top 5 region-to-gene links per gene:   1%|          | 13/2349 [00:00<01:39, 23.49it/s]
2;1mProcessing: negative r2g, Top 5 region-to-gene links per gene:   1%|          | 17/2349 [00:00<01:29, 25.99it/s]
2;1mProcessing: negative r2g, Top 5 region-to-gene links per gene:   1%|          | 20/2349 [0

Processing: negative r2g, 0.9 quantile:  56%|█████▌    | 1417/2529 [01:21<00:43, 25.51it/s]


Processing: negative r2g, 0.9 quantile:  80%|███████▉  | 2021/2529 [01:43<00:22, 22.99it/s]


Processing: 0.95 quantile, negative r2g:   8%|▊         | 213/2529 [00:06<00:59, 39.00it/s]


...

2025-01-12 08:29:16,700 GSEA         INFO     Subsetting TF2G adjacencies for TF with motif.


2025-01-12 08:29:34,032	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


2025-01-12 08:29:35,061 GSEA         INFO     Running GSEA...


Running using 40 cores: 100%|██████████| 60984/60984 [1:08:18<00:00, 14.88it/s]   


2025-01-12 10:06:54,803 GSEA         INFO     Subsetting on adjusted pvalue: 1, minimal NES: 0 and minimal leading edge genes 10
2025-01-12 10:07:28,293 GSEA         INFO     Merging eRegulons
2025-01-12 10:07:42,843 GSEA         INFO     Storing eRegulons in .uns[eRegulons_importance].
2025-01-12 10:09:55:End build GRN.


# 6. Format egrn to get eRegulon metadata and signature 

## 6A. Generate eRegulon metadata

In [32]:
from scenicplus.utils import format_egrns
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':Start.',sep="")

format_egrns(
    scplus_obj, 
    eregulons_key = 'eRegulons_importance', 
    TF2G_key = 'TF2G_adj', 
    key_added = 'eRegulon_metadata'
)
print(scplus_obj.uns['eRegulon_metadata'][0:10])

print("Done! Generate eRegulon metadata!")
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':End.',sep="")

2026-06-14 09:23:12:Start.
  Region_signature_name Gene_signature_name    TF is_extended  \
0        ATF1_+_+_(41r)      ATF1_+_+_(35g)  ATF1       False   
1        ATF1_+_+_(41r)      ATF1_+_+_(35g)  ATF1       False   
2        ATF1_+_+_(41r)      ATF1_+_+_(35g)  ATF1       False   
3        ATF1_+_+_(41r)      ATF1_+_+_(35g)  ATF1       False   
4        ATF1_+_+_(41r)      ATF1_+_+_(35g)  ATF1       False   
5        ATF1_+_+_(41r)      ATF1_+_+_(35g)  ATF1       False   
6        ATF1_+_+_(41r)      ATF1_+_+_(35g)  ATF1       False   
7        ATF1_+_+_(41r)      ATF1_+_+_(35g)  ATF1       False   
8        ATF1_+_+_(41r)      ATF1_+_+_(35g)  ATF1       False   
9        ATF1_+_+_(41r)      ATF1_+_+_(35g)  ATF1       False   

                     Region     Gene  R2G_importance   R2G_rho  \
0   chr19:51311699-51312199     NKG7        0.059735  0.014761   
1  chr1:100894774-100895274     DPH5        0.064928  0.059208   
2   chr20:53611664-53612164   ZNF217        0.043815  0.072

## 6B. Format eRegulons

In [42]:
from scenicplus.eregulon_enrichment import *

print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':Start.',sep="")
get_eRegulons_as_signatures(
    scplus_obj, 
    eRegulon_metadata_key='eRegulon_metadata', 
    key_added='eRegulon_signatures'
)

print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':End.',sep="")
print("Done! Generate eRegulon signatures!")


2026-06-14 09:52:51:Start.
2026-06-14 09:53:44:End.
Done! Generate eRegulon signatures!


In [44]:
scplus_obj.uns['eRegulon_signatures'].keys()

dict_keys(['Gene_based', 'Region_based'])

In [47]:
import dill
with open(f'{scplus_dir}/01.scplus_obj.pkl', 'wb') as f:
    dill.dump(scplus_obj, f) 

# 7.score_eRegulons

## 7A. Gene based

In [48]:
# Gene based raking
from scenicplus.eregulon_enrichment import *
from scenicplus.cistromes import *
import time

print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':Start gene_ranking',sep="")
gene_ranking = make_rankings(scplus_obj, target='gene')
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':Done',sep="")


2026-06-14 10:40:44:Start gene_ranking
2026-06-14 10:41:22:Done


In [50]:
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':Start',sep="")
auc_gene = signature_enrichment(
    rankings = gene_ranking,
    signatures = scplus_obj.uns['eRegulon_signatures']['Gene_based'],
    enrichment_type='gene',
    auc_threshold = 0.05,
    normalize = False,
    n_cpu = 40
)
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':Done',sep="")

print(auc_gene.shape)
auc_gene.head()

2026-06-14 11:28:41:Start
2026-06-14 11:29:28:Done
(9856, 1851)


,GABPA_extended_-_-_(53g),ZNF2_extended_-_-_(18g),CARF_+_+_(20g),ZGPAT_extended_+_+_(81g),TCF7L1_extended_-_+_(55g),SOX5_extended_+_-_(1236g),RBAK_extended_-_-_(17g),ZBTB7A_extended_-_+_(152g),ZBTB25_extended_+_+_(44g),ARID5A_extended_-_-_(15g),...,CUX1_extended_-_-_(65g),ZNF273_extended_+_+_(120g),ZNF415_+_-_(20g),CREB5_extended_+_+_(155g),DLX5_extended_+_+_(79g),FOXN3_+_-_(29g),ZNF33B_+_-_(11g),TFDP2_+_+_(99g),TAL1_extended_-_+_(11g),ZNF180_-_+_(25g)
Cell,,,,,,,,,,,,,,,,,,,,,
TGTGGCGGTAATCCCT-1_BICCN-set1-10___human_DG,0.058326,0.045143,0.000000,0.013034,0.049230,0.133885,0.000000,0.069410,0.056309,0.117996,...,0.033720,0.006444,0.002596,0.154303,0.008273,0.001489,0.047740,0.010383,0.018381,0.039126
GACTATTCAGCTACGT-1_BICCN-set1-7___human_DG,0.045427,0.076260,0.038470,0.010423,0.009886,0.139234,0.036419,0.051179,0.009799,0.078470,...,0.065364,0.020961,0.000000,0.204410,0.005990,0.000000,0.006160,0.025186,0.000000,0.031169
CAAGGTAAGGCCTGGT-1_BICCN-set1-1___human_DG,0.004310,0.093139,0.085710,0.027484,0.025643,0.042783,0.000000,0.079350,0.016704,0.143607,...,0.018041,0.006689,0.056585,0.292505,0.018911,0.000000,0.047193,0.017464,0.074516,0.000000
ATACCGGTCGCTATAA-1_BICCN-set1-7___human_DG,0.014641,0.041803,0.055792,0.018519,0.015390,0.033904,0.037673,0.051934,0.049205,0.122878,...,0.024372,0.015893,0.000000,0.276221,0.000000,0.053646,0.000000,0.021748,0.000000,0.045464
CCTGACTTCTAATTGG-1_BICCN-set1-1___human_DG,0.015259,0.000000,0.000000,0.025575,0.039513,0.034780,0.000000,0.029159,0.035345,0.000000,...,0.016780,0.012427,0.062869,0.237314,0.008473,0.007971,0.000000,0.014274,0.056384,0.032459


In [51]:
auc_gene.to_csv(f'{scplus_dir}/07.eRegulon_AUC_Gene_based.csv') 

## 7B. Region based

In [56]:
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':Start region_ranking',sep="")
region_ranking = make_rankings(scplus_obj, target='region')
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':Done',sep="")

2026-06-14 11:54:41:Start region_ranking
2026-06-14 12:09:24:Done


In [57]:
# Score region regulons
print("score_eRegulons for region based")
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':Start',sep="")
score_eRegulons(
    scplus_obj,
    ranking = region_ranking,
    eRegulon_signatures_key = 'eRegulon_signatures',
    key_added = 'eRegulon_AUC',
    enrichment_type= 'region',
    auc_threshold = 0.05,
    normalize = False,
    n_cpu = 40
)
print(time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()), ':Done',sep="")

auc_region = scplus_obj.uns['eRegulon_AUC']['Region_based']
print(auc_region.shape)
auc_region.head()

score_eRegulons for region based
2026-06-14 12:09:24:Start
2026-06-14 12:11:25:Done
(9856, 1851)


,ZBTB7B_-_-_(27r),ZNF236_extended_-_-_(16r),ZNF350_extended_+_+_(28r),MEIS1_extended_-_+_(31r),TFDP1_-_-_(245r),SNAI1_+_+_(10r),ZIC2_+_+_(15r),TCF4_extended_-_+_(62r),LHX3_extended_+_-_(30r),LEF1_+_+_(119r),...,ZNF205_extended_+_+_(27r),POU3F2_-_-_(161r),ZNF423_-_-_(32r),ZNF3_extended_+_+_(36r),FOSB_extended_-_-_(28r),FOSL2_-_-_(348r),ZNF200_extended_-_+_(77r),ZNF708_extended_+_-_(19r),IRF4_extended_+_-_(171r),FOXC1_+_+_(141r)
Cell,,,,,,,,,,,,,,,,,,,,,
TGTGGCGGTAATCCCT-1_BICCN-set1-10___human_DG,0.469379,0.628115,0.307717,0.210469,0.522774,0.135242,0.096271,0.156516,0.391088,0.118498,...,0.240974,0.313798,0.475198,0.192412,0.343800,0.171972,0.287729,0.068215,0.358976,0.347999
GACTATTCAGCTACGT-1_BICCN-set1-7___human_DG,0.381795,0.514670,0.248991,0.178970,0.428036,0.067162,0.127807,0.109489,0.360327,0.111020,...,0.249790,0.277327,0.409367,0.095337,0.335473,0.160128,0.223050,0.055372,0.328202,0.283635
CAAGGTAAGGCCTGGT-1_BICCN-set1-1___human_DG,0.284864,0.468725,0.291680,0.089094,0.424862,0.063971,0.141432,0.142489,0.409493,0.072834,...,0.250263,0.303769,0.450506,0.011500,0.290252,0.144757,0.204945,0.021413,0.322231,0.192944
ATACCGGTCGCTATAA-1_BICCN-set1-7___human_DG,0.429691,0.618887,0.387445,0.161073,0.552213,0.000000,0.216190,0.183980,0.438672,0.082854,...,0.318666,0.343713,0.573061,0.044093,0.291854,0.176253,0.267071,0.024406,0.358216,0.272253
CCTGACTTCTAATTGG-1_BICCN-set1-1___human_DG,0.318124,0.466257,0.267468,0.137079,0.419639,0.054819,0.156777,0.139286,0.420331,0.071133,...,0.257773,0.286918,0.452442,0.015131,0.315161,0.144245,0.214399,0.031403,0.324401,0.220323


In [58]:
auc_region.to_csv(f'{scplus_dir}/07.eRegulon_AUC_Region_based.csv') 

# 8. Generate anndata

In [15]:
import anndata as ad

auc_gene.columns = [x.split('_(')[0] for x in auc_gene.columns]
adata = ad.AnnData(
    auc_gene,
    obs=rna.obs.loc[auc_gene.index]
)

adata

AnnData object with n_obs × n_vars = 9856 × 1851
    obs: 'class', 'subclass', 'subtype'

In [16]:
adata.write_h5ad(f'{scplus_dir}/08.eRegulon_AUC_Gene.h5ad') 